In [1]:
# PINN CUSTOM LOSS FUNCTION - tuned for Colab - V.P. Carey, ME249 F25

import tensorflow as tf
from tensorflow.keras import backend as K

def custom_loss_function(kappa, x_max, y_max, epoch_tracker):
    """ Custom loss function for the neural network model.
        - kappa: weight for the Laplace residual term.
        - x_max, y_max: maximum x and y coordinates of the domain.
        - epoch_tracker: callback object to keep track of the current epoch. """
    
    def loss(y_true, y_pred):    
        # Define pointsBoundaryChk as a constant tensor
        pointsBoundaryChk = tf.constant([
            [ 3.0, 0.0 ], 
            [ 3.0, 0.5 ], 
            [ 3.0, 1.0 ], 
            [ 3.0, 1.5 ],
            [ 3.0, 2.0 ], 
            [ 3.0, 2.5 ],
            [ 3.0, 3.0 ], 
            [ 3.0, 3.5 ], 
            # ADD TO THIS LIST TO INCLUDE ALL 32 [X,Y] POINTS IN pointsBoundaryChk ARRAY IN PROJECT WRITEUP 
            [ 1.0, 5.0 ],
            [ 1.5, 5.0 ],
            [ 2.0, 5.0 ],
            [ 2.5, 5.0 ]
        ], dtype=tf.float32)
        
        # insert
        # 1. Boundary loss: Calculate squared error for the boundary condition
        # Ensure y_pred and y_true have static shapes using tf.shape
        y_pred = tf.reshape(y_pred, tf.shape(y_true))  # Reshape y_pred to match y_true's shape
        boundary_loss = K.mean(K.square(y_pred - y_true))  # Calculate boundary loss
        
        
        #=======
        # Use tf.where for conditional logic to ensure consistent shape
        ESbdsum = 0.
        for _ in range(32):
            idx = tf.random.uniform(shape=[], minval=0, maxval=32, dtype=tf.int32)
            selected_point = tf.gather(pointsBoundaryChk, idx)
            input_tensor = tf.expand_dims(selected_point, axis=0)
            T_pred_at_boundxy = model(input_tensor)  

            # Replacing tf.cond with tf.where to ensure a consistent shape
            ESbdsum += tf.where(T_pred_at_boundxy < 5., 20.*(5. - T_pred_at_boundxy)*(5. - T_pred_at_boundxy), 0.) 
            ESbdsum += tf.where(T_pred_at_boundxy > 55., 20.*(T_pred_at_boundxy - 55.)*(T_pred_at_boundxy - 55.), 0.)
            
        boundary_loss += ESbdsum/32.
        #=======
    
        # 2. Check epoch from the callback
        epoch = epoch_tracker.epochs_trained
            
        # If current epoch is less than N_epInit, ignore the Laplace residual
        if epoch < epoch_tracker.N_epInit:
            total_loss = boundary_loss
        else:
            # 3. Randomly select an interior point from this list using TensorFlow's random uniform
            points = tf.constant([
                [0.5, 0.5],
                [0.5, 1.0],
                [0.5, 1.5],
                [0.5, 2.],
                [0.5, 2.5],
                [0.5, 3.],
                # ADD TO THIS LIST TO INCLUDE ALL 45 [X,Y] POINTS IN points ARRAY IN PROJECT WRITEUP
                [2.5, 0.5],
                [2.5, 1.0],
                [2.5, 1.5],
                [2.5, 2.],
                [2.5, 2.5],
                [2.5, 3.],
                [2.5, 3.5],
                [2.5, 4.],
                [2.5, 4.5]
            ], dtype=tf.float32)
            
            meanLap_res = 0.
            meanTlimPen = 0.
            for _ in range(45):
                ##compute Laplacian residue here - need to do this once per BC point - include in range 32 loop =======
                # Use tf.random.uniform to sample an index for the interior point
                idx = tf.random.uniform(shape=[], minval=0, maxval=45, dtype=tf.int32)
                selected_point = tf.gather(points, idx)
                x_interior, y_interior = selected_point[0], selected_point[1],
                
                # 4. Use tf.GradientTape to compute second derivatives
                with tf.GradientTape(persistent=True) as tape:
                    tape.watch(selected_point)
                    # Pass the selected (x_interior, y_interior) to the model to get T_pred_at_interior
                    T_pred_at_interior = model([x_interior, y_interior])  # Model prediction
                
                # Compute first derivatives with respect to x and y
                dT_dx = tape.gradient(T_pred_at_interior, selected_point[0])
                dT_dy = tape.gradient(T_pred_at_interior, selected_point[1])
                
                # Compute second derivatives (Hessian)\n",
                d2T_dx2 = tape.gradient(dT_dx, selected_point[0])
                d2T_dy2 = tape.gradient(dT_dy, selected_point[1])
                
                # Calculate the Laplacian (sum of second derivatives)
                laplacian = d2T_dx2 + d2T_dy2
                
                # Compute the Laplacian residual (sum of squared Laplacian)
                #meanLap_res +=  K.square(laplacian)/32.
                # Calculate the Laplacian (sum of second derivatives)
                laplacian = d2T_dx2 + d2T_dy2

                # Compute the Laplacian residual (sum of squared Laplacian)
                # Ensure laplacian has a defined shape before adding it to meanLap_res
                laplacian = tf.reshape(laplacian, []) # Reshape to a scalar
                meanLap_res += K.square(laplacian) / 45.
        
                #insert
                # Compute the Laplacian residual (sum of squared Laplacian)
                # Ensure laplacian has a defined shape before adding it to meanLap_res
                laplacian = tf.reshape(laplacian, []) # Reshape to a scalar
                meanLap_res += K.square(laplacian) / 45.
                
                # 4b. NEW: ADD PENALTY IF T below 5 or above 55
            
                # Replace tf.cond with tf.where inside the inner loop
                meanTlimPen += tf.where(T_pred_at_interior < 5., 20.*(5. - T_pred_at_interior)*(5. - T_pred_at_interior)/45., 0.)
                meanTlimPen += tf.where(T_pred_at_interior > 55., 20.*(T_pred_at_interior - 55.)*(T_pred_at_interior - 55.)/45., 0.)
            
            # 5. Combine three losses
            total_loss = boundary_loss + kappa * meanLap_res + meanTlimPen

    
        return total_loss
    return loss
    
    


2025-11-25 10:26:11.326247: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-25 10:26:11.329810: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-25 10:26:11.341221: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-11-25 10:26:11.359699: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-11-25 10:26:11.365169: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-11-25 10:26:11.378621: I tensorflow/core/platform/cpu_feature_gu